# 16. Building an API for the Fine-Tuned Model

To let **other apps** (a website, a mobile app) use your model, you wrap it in an **API** -
a web address they can send a question to and get an answer back. We use **FastAPI**.

## Real-life analogy

Your model is a **chef**. The API is the **waiter**: apps give their order to the waiter,
the waiter asks the chef, and brings the food back. Apps never touch the kitchen directly.

## How it works

| Piece | Job |
|-------|-----|
| **FastAPI** | Creates the web server and the `/generate` address |
| **Load model once** | Load the model at startup (not on every request - that would be slow) |
| **Endpoint** | Receives a question, runs the model, returns the answer |
| **uvicorn** | Runs the server |

## Step 1 - Write the API to a file (`app.py`)

We save it as a file because an API is run as a **program**, not inside a notebook cell.

In [1]:
api_code = '''
from fastapi import FastAPI
from pydantic import BaseModel
from transformers import pipeline

app = FastAPI()

# Load the model ONCE when the server starts (fast for every request after).
chat = pipeline("text-generation", model="merged_model")

# The shape of the incoming request: {"question": "..."}
class Question(BaseModel):
    question: str

@app.post("/generate")
def generate(item: Question):
    messages = [{"role": "user", "content": item.question}]
    out = chat(messages, max_new_tokens=50)
    answer = out[0]["generated_text"][-1]["content"]
    return {"answer": answer}
'''

with open("app.py", "w", encoding="utf-8") as f:
    f.write(api_code)
print("Saved app.py")

Saved app.py


## Step 2 - Run the server

Open a terminal (not the notebook) in this folder and run:

```bash
uvicorn app:app --reload
```

Then open **http://127.0.0.1:8000/docs** in your browser to try it - FastAPI gives you a test page automatically.

## Step 3 - Call the API from another program

In [ ]:
# Run this AFTER the server is running (in a separate terminal).
# It sends a question to your API and prints the answer.
import requests

response = requests.post(
    "http://127.0.0.1:8000/generate",
    json={"question": "How long does delivery take?"},
)
print(response.json())

## Key points to remember

- An **API** lets other apps use your model over the web.
- **FastAPI** builds the server; **uvicorn** runs it (`uvicorn app:app --reload`).
- **Load the model once** at startup, not on every request (speed!).
- Test easily at **`/docs`** - FastAPI makes an interactive page for you.
- Other apps call it with a simple **POST** request and get JSON back.
- This is the final step: your fine-tuned model is now a **real service**! 🎉